In [1]:
# IMPORTS

import json
import os
import pickle
import random
from datetime import datetime, timezone
from pathlib import Path

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# PyTorch TabNet
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

# TensorFlow / Keras
import tensorflow as tf

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from tensorflow.keras.models import Sequential, Model

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization,
    Bidirectional,
    LSTM,
    GRU,
    Input,
    Reshape,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D
)

from tensorflow.keras.callbacks import EarlyStopping


def fraud_metrics():
    return [
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    ]

import warnings
warnings.filterwarnings("ignore")

In [2]:
# PATHS

root_path = Path.cwd().resolve()
if root_path.name == "notebooks":
    root_path = root_path.parent

data_dir = root_path / "data"
artifact_dir = root_path / "artifacts"
candidate_model_dir = artifact_dir / "model_candidates"
model_dir = root_path / "models"
result_dir = root_path / "results"

for directory in [data_dir, artifact_dir, candidate_model_dir, model_dir, result_dir]:
    directory.mkdir(parents=True, exist_ok=True)

raw_data_path = data_dir / "downsampled_transactions.csv"
preprocessing_bundle_path = artifact_dir / "preprocessing.pkl"
training_manifest_path = artifact_dir / "training_manifest.json"
thresholds_path = artifact_dir / "decision_thresholds.json"

for candidate_artifact in candidate_model_dir.glob("*"):
    if candidate_artifact.is_file():
        candidate_artifact.unlink()


In [3]:
# PREPROCESSING HELPERS

FEATURE_COLS = [
    "log_amount",
    "error_orig",
    "error_dest",
    "hour",
    "is_night",
    "is_high_amount",
    "type_encoded",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

TYPE_MAPPING = {"TRANSFER": 0, "CASH_OUT": 1}
RAW_NUMERIC_COLS = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

# Cleaning raw transactions
def clean_raw_transactions(df):
    df = df.copy()

    for col in RAW_NUMERIC_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "type" not in df.columns:
        raise ValueError("Missing required column: type")

    if "isFraud" in df.columns:
        df["isFraud"] = pd.to_numeric(df["isFraud"], errors="coerce").astype("Int64")

    df = df.dropna(subset=["type"] + RAW_NUMERIC_COLS)
    return df

# Fitting preprocessing metadata
def fit_preprocessing_metadata(df):
    df = clean_raw_transactions(df)
    supported = df[df["type"].isin(TYPE_MAPPING)]

    if supported.empty:
        raise ValueError("No TRANSFER or CASH_OUT rows available for training.")

    return {
        "type_mapping": TYPE_MAPPING.copy(),
        "high_amount_threshold": float(supported["amount"].quantile(0.99)),
    }

# Binary indicator conversion
def _binary_indicator(series):
    numeric = pd.to_numeric(series, errors="coerce")
    text = series.astype(str).str.strip().str.lower()
    return numeric.fillna(text.isin(["true", "yes", "1"]).astype(int)).astype(int)

# Class distribution dictionary
def class_distribution_dict(series):
    counts = series.value_counts(dropna=False).sort_index().astype(int).to_dict()
    return {str(k): int(v) for k, v in counts.items()}

# Engineering features
def engineer_features(df, high_amount_threshold, type_mapping=None):
    df = clean_raw_transactions(df)
    type_mapping = TYPE_MAPPING if type_mapping is None else type_mapping

    df = df[df["type"].isin(type_mapping)].copy()

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
    df["error_orig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    df["error_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    df["hour"] = (df["step"] % 24).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    df["is_high_amount"] = (df["amount"] > high_amount_threshold).astype(int)
    df["type_encoded"] = df["type"].map(type_mapping).astype(int)

    if "nameDest" in df.columns:
        df["is_merchant_dest"] = df["nameDest"].astype(str).str.startswith("M").astype(int)
    elif "is_merchant_dest" in df.columns:
        df["is_merchant_dest"] = _binary_indicator(df["is_merchant_dest"])
    else:
        df["is_merchant_dest"] = 0

    for col in FEATURE_COLS + ["is_merchant_dest"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


In [4]:
# LOAD DATA

if not raw_data_path.exists():
    raise FileNotFoundError("Missing data/downsampled_transactions.csv. Run preprocessing_pipeline.ipynb before model_training.ipynb.")

df = pd.read_csv(raw_data_path)
df = clean_raw_transactions(df)
df = df[df["type"].isin(TYPE_MAPPING)].copy()

invalid_label_mask = ~df["isFraud"].isin([0, 1])
if invalid_label_mask.any():
    raise ValueError(f"isFraud must contain only 0/1 labels; found {int(invalid_label_mask.sum())} invalid rows.")
df["isFraud"] = df["isFraud"].astype(int)

if df["isFraud"].nunique() < 2:
    raise ValueError("Training requires both fraud and non-fraud examples.")

training_row_count = int(len(df))
training_class_distribution = class_distribution_dict(df["isFraud"])

print("Rows used after supported-type filtering:", len(df))
print("Training class distribution:", training_class_distribution)
print("Class distribution:", df["isFraud"].value_counts().sort_index().to_dict())
print("Fraud positive rate:", round(float(df["isFraud"].mean()), 6))


Rows used after supported-type filtering: 49278
Training class distribution: {'0': 41065, '1': 8213}
Class distribution: {0: 41065, 1: 8213}
Fraud positive rate: 0.166667


In [5]:
# TRAIN / VALIDATION / TEST SPLIT

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=RANDOM_SEED,
    stratify=df["isFraud"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_df["isFraud"],
)

preprocessing_metadata = fit_preprocessing_metadata(train_df)
preprocessing_metadata.update(
    {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_data": str(raw_data_path.relative_to(root_path)),
        "training_row_count": training_row_count,
        "training_class_distribution": training_class_distribution,
        "dataset_source": "preprocessing_pipeline_downsampled_transactions",
        "feature_columns": FEATURE_COLS,
        "target_column": "isFraud",
        "split_strategy": "stratified_train_validation_test",
        "split_ratios": {"train": 0.7, "validation": 0.15, "test": 0.15},
        "imbalance_strategy": "pre_downsampled_dataset_with_balanced_class_weights",
    }
)

train_df = engineer_features(
    train_df,
    preprocessing_metadata["high_amount_threshold"],
    preprocessing_metadata["type_mapping"],
)
val_df = engineer_features(
    val_df,
    preprocessing_metadata["high_amount_threshold"],
    preprocessing_metadata["type_mapping"],
)
test_df = engineer_features(
    test_df,
    preprocessing_metadata["high_amount_threshold"],
    preprocessing_metadata["type_mapping"],
)

features = FEATURE_COLS
X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]
y_train = train_df["isFraud"].astype(int)
y_val = val_df["isFraud"].astype(int)
y_test = test_df["isFraud"].astype(int)

split_profile = {
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "train_class_distribution": class_distribution_dict(y_train),
    "validation_class_distribution": class_distribution_dict(y_val),
    "test_class_distribution": class_distribution_dict(y_test),
}

print(split_profile)


{'train_rows': 34494, 'validation_rows': 7392, 'test_rows': 7392, 'train_class_distribution': {'0': 28745, '1': 5749}, 'validation_class_distribution': {'0': 6160, '1': 1232}, 'test_class_distribution': {'0': 6160, '1': 1232}}


In [6]:
# SCALING AND IMBALANCE HANDLING

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train,
)
class_weight_dict = {0: float(class_weights[0]), 1: float(class_weights[1])}
class_distribution = class_distribution_dict(y_train)
positive_rate = float(y_train.mean())

preprocessing_bundle = {
    "scaler": scaler,
    "metadata": preprocessing_metadata,
    "feature_cols": features,
    "class_weight": class_weight_dict,
    "class_distribution": class_distribution,
    "positive_rate": positive_rate,
    "split_profile": split_profile,
    "imbalance_strategy": "full_dataset_with_balanced_class_weights",
    "decision_thresholds": {},
}

with open(preprocessing_bundle_path, "wb") as f:
    pickle.dump(preprocessing_bundle, f)

print("Feature columns:", features)
print("Training class distribution:", class_distribution)
print("Training fraud positive rate:", round(positive_rate, 6))
print("Class weights:", class_weight_dict)


Feature columns: ['log_amount', 'error_orig', 'error_dest', 'hour', 'is_night', 'is_high_amount', 'type_encoded', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Training class distribution: {'0': 28745, '1': 5749}
Training fraud positive rate: 0.166667
Class weights: {0: 0.6, 1: 3.0}


In [7]:
# THRESHOLD SELECTION HELPERS

MIN_RECALL_FOR_FRAUD = 0.80


def select_operating_threshold(y_true, y_prob, min_recall=MIN_RECALL_FOR_FRAUD):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    candidates = []

    for threshold, p, r in zip(thresholds, precision[:-1], recall[:-1]):
        f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
        candidates.append(
            {
                "threshold": float(threshold),
                "precision": float(p),
                "recall": float(r),
                "f1": float(f1),
            }
        )

    eligible = [row for row in candidates if row["recall"] >= min_recall]
    if eligible:
        return max(eligible, key=lambda row: (row["f1"], row["precision"], row["recall"]))

    return max(candidates, key=lambda row: (row["f1"], row["recall"], row["precision"]))


model_thresholds = {}


def finalize_model_outputs(name, val_prob, test_prob):
    threshold_info = select_operating_threshold(y_val.to_numpy(dtype=int), val_prob)
    threshold = threshold_info["threshold"]
    model_thresholds[name] = threshold_info
    test_pred = (test_prob >= threshold).astype(int)
    print(f"{name} threshold:", threshold_info)
    return test_pred


In [8]:
# MODEL 1: MLP (MULTI-LAYER PERCEPTRON)

mlp = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

mlp.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=fraud_metrics()
)

# Early stopping
es = EarlyStopping(
    monitor='val_pr_auc',
    patience=3,
    restore_best_weights=True,
    mode='max'
)

# Training
mlp_history = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[es]
)

# Predictions
mlp_val_prob = mlp.predict(X_val, verbose=0).ravel()
mlp_prob = mlp.predict(X_test, verbose=0).ravel()
mlp_pred = finalize_model_outputs("MLP", mlp_val_prob, mlp_prob)

mlp.save(candidate_model_dir / "mlp_model.keras")

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 0.3394 - pr_auc: 0.8126 - precision: 0.5500 - recall: 0.8504 - roc_auc: 0.9287 - val_loss: 0.3102 - val_pr_auc: 0.9208 - val_precision: 0.8242 - val_recall: 0.8677 - val_roc_auc: 0.9719
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1899 - pr_auc: 0.9263 - precision: 0.7316 - recall: 0.9148 - roc_auc: 0.9780 - val_loss: 0.1545 - val_pr_auc: 0.9300 - val_precision: 0.9210 - val_recall: 0.7849 - val_roc_auc: 0.9755
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1426 - pr_auc: 0.9544 - precision: 0.7813 - recall: 0.9400 - roc_auc: 0.9873 - val_loss: 0.1133 - val_pr_auc: 0.9511 - val_precision: 0.9621 - val_recall: 0.8028 - val_roc_auc: 0.9838
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1158 - pr_auc: 0.9673 - precision: 0.8206 - recall: 0.9541 - roc_auc: 0.9911 - val_loss: 0.0824 - val_pr_auc: 0.9721 - val_precision: 0.9606 - val_recall: 0.8709 - val_roc_auc: 0.9911
Epoch 5/20
1

In [9]:
# SEQUENCE RESHAPE

# Preserve the original tabular models while adapting sequence-style architectures to the same feature set.
X_train_seq = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val_seq = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
X_test_seq = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))


In [10]:
# MODEL 2: GRU (GATED RECURRENT UNIT)

gru = Sequential([
    GRU(64, input_shape=(1, X_train.shape[1]), return_sequences=False),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

gru.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=fraud_metrics()
)

# Early stopping
es = EarlyStopping(
    monitor='val_pr_auc',
    patience=3,
    restore_best_weights=True,
    mode='max'
)

# Training
gru_history = gru.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[es]
)

# Predictions
gru_val_prob = gru.predict(X_val_seq, verbose=0).ravel()
gru_prob = gru.predict(X_test_seq, verbose=0).ravel()
gru_pred = finalize_model_outputs("GRU", gru_val_prob, gru_prob)

gru.save(candidate_model_dir / "gru_model.keras")

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - loss: 0.4482 - pr_auc: 0.7374 - precision: 0.6017 - recall: 0.7344 - roc_auc: 0.8856 - val_loss: 0.2595 - val_pr_auc: 0.8807 - val_precision: 0.7366 - val_recall: 0.8352 - val_roc_auc: 0.9540
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.2388 - pr_auc: 0.9096 - precision: 0.7518 - recall: 0.8728 - roc_auc: 0.9677 - val_loss: 0.1828 - val_pr_auc: 0.9424 - val_precision: 0.7810 - val_recall: 0.9058 - val_roc_auc: 0.9809
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1826 - pr_auc: 0.9407 - precision: 0.7717 - recall: 0.9123 - roc_auc: 0.9810 - val_loss: 0.1424 - val_pr_auc: 0.9559 - val_precision: 0.8169 - val_recall: 0.9164 - val_roc_auc: 0.9857
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1569 - pr_auc: 0.9535 - precision: 0.7910 - recall: 0.9301 - roc_auc: 0.9857 - val_loss: 0.1303 - val_pr_auc: 0.9639 - val_precision: 0.8240 - val_recall: 0.9310 - val_roc_auc: 0.9884
Epoch 5/20
135

In [11]:
# MODEL 3: BiLSTM (BIDIRECTIONAL LONG SHORT-TERM MEMORY)

bilstm = Sequential([
    Bidirectional(
        LSTM(64, return_sequences=False),
        input_shape=(1, X_train.shape[1])
    ),

    Dropout(0.3),

    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

bilstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=fraud_metrics()
)

# Early stopping
es = EarlyStopping(
    monitor='val_pr_auc',
    patience=3,
    restore_best_weights=True,
    mode='max'
)

# Training
bilstm_history = bilstm.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[es]
)

# Predictions
bilstm_val_prob = bilstm.predict(X_val_seq, verbose=0).ravel()
bilstm_prob = bilstm.predict(X_test_seq, verbose=0).ravel()
bilstm_pred = finalize_model_outputs("BiLSTM", bilstm_val_prob, bilstm_prob)

bilstm.save(candidate_model_dir / "bilstm_model.keras")

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - loss: 0.4459 - pr_auc: 0.7510 - precision: 0.5493 - recall: 0.7822 - roc_auc: 0.8912 - val_loss: 0.2503 - val_pr_auc: 0.8900 - val_precision: 0.7577 - val_recall: 0.8425 - val_roc_auc: 0.9583
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.2211 - pr_auc: 0.9223 - precision: 0.7765 - recall: 0.8833 - roc_auc: 0.9729 - val_loss: 0.1702 - val_pr_auc: 0.9455 - val_precision: 0.7901 - val_recall: 0.9075 - val_roc_auc: 0.9817
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.1685 - pr_auc: 0.9499 - precision: 0.7911 - recall: 0.9181 - roc_auc: 0.9837 - val_loss: 0.1383 - val_pr_auc: 0.9598 - val_precision: 0.8184 - val_recall: 0.9253 - val_roc_auc: 0.9867
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1449 - pr_auc: 0.9595 - precision: 0.8049 - recall: 0.9367 - roc_auc: 0.9877 - val_loss: 0.1182 - val_pr_auc: 0.9680 - val_precision: 0.8459 - val_recall: 0.9310 - val_roc_auc: 0.9894
Epoch 5/20
1

In [12]:
# MODEL 4: TABNET

tabnet = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=RANDOM_SEED,
)

# Training
tabnet.fit(
    X_train, y_train.values,
    eval_set=[(X_val, y_val.values)],
    eval_metric=['logloss', 'auc'],
    max_epochs=30,
    patience=5,
    batch_size=1024,
    virtual_batch_size=128,
    weights=class_weight_dict,
)

# Predictions
tabnet_val_prob = tabnet.predict_proba(X_val)[:, 1]
tabnet_prob = tabnet.predict_proba(X_test)[:, 1]
tabnet_pred = finalize_model_outputs("TabNet", tabnet_val_prob, tabnet_prob)

tabnet.save_model(str(candidate_model_dir / "tabnet_model"))


epoch 0  | loss: 0.35757 | val_0_logloss: 0.46312 | val_0_auc: 0.96694 |  0:00:06s
epoch 1  | loss: 0.1895  | val_0_logloss: 0.16799 | val_0_auc: 0.98262 |  0:00:10s
epoch 2  | loss: 0.14164 | val_0_logloss: 0.13675 | val_0_auc: 0.98321 |  0:00:28s
epoch 3  | loss: 0.13139 | val_0_logloss: 0.148   | val_0_auc: 0.9876  |  0:00:32s
epoch 4  | loss: 0.12083 | val_0_logloss: 0.2019  | val_0_auc: 0.9835  |  0:00:36s
epoch 5  | loss: 0.11931 | val_0_logloss: 0.27739 | val_0_auc: 0.97735 |  0:00:42s
epoch 6  | loss: 0.11662 | val_0_logloss: 0.52348 | val_0_auc: 0.87001 |  0:00:46s
epoch 7  | loss: 0.11362 | val_0_logloss: 0.38825 | val_0_auc: 0.90758 |  0:00:51s
epoch 8  | loss: 0.10343 | val_0_logloss: 0.24067 | val_0_auc: 0.98118 |  0:00:55s

Early stopping occurred at epoch 8 with best_epoch = 3 and best_val_0_auc = 0.9876
TabNet threshold: {'threshold': 0.8536798357963562, 'precision': 0.909704641350211, 'recall': 0.875, 'f1': 0.8920148944973108}
Successfully saved model at C:\Project\Tra

'C:\\Project\\Transactional-Fraud-Detection\\artifacts\\model_candidates\\tabnet_model.zip'

In [13]:
# MODEL 5: FT-TRANSFORMER

inputs = Input(shape=(X_train.shape[1],))

x = Reshape((X_train.shape[1], 1))(inputs)

x = Dense(128)(x)

attn = MultiHeadAttention(
    num_heads=4,
    key_dim=32
)(x, x)

x = LayerNormalization()(x + attn)

ffn = Dense(256, activation="relu")(x)
ffn = Dropout(0.2)(ffn)
ffn = Dense(128)(ffn)

x = LayerNormalization()(x + ffn)

x = GlobalAveragePooling1D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)

outputs = Dense(1, activation='sigmoid')(x)

ft_transformer = Model(inputs, outputs)

ft_transformer.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=fraud_metrics()
)

# Early stopping
es = EarlyStopping(
    monitor='val_pr_auc',
    patience=3,
    restore_best_weights=True,
    mode='max'
)

# Training
ft_history = ft_transformer.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[es]
)

# Predictions
ft_transformer_val_prob = ft_transformer.predict(X_val, verbose=0).ravel()
ft_transformer_prob = ft_transformer.predict(X_test, verbose=0).ravel()
ft_transformer_pred = finalize_model_outputs("FT-Transformer", ft_transformer_val_prob, ft_transformer_prob)

ft_transformer.save(candidate_model_dir / "ft_transformer_model.keras")

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 26s 143ms/step - loss: 0.5001 - pr_auc: 0.5585 - precision: 0.3909 - recall: 0.7441 - roc_auc: 0.8358 - val_loss: 0.4262 - val_pr_auc: 0.6496 - val_precision: 0.4572 - val_recall: 0.7679 - val_roc_auc: 0.8735
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 19s 132ms/step - loss: 0.4525 - pr_auc: 0.6451 - precision: 0.4417 - recall: 0.7734 - roc_auc: 0.8687 - val_loss: 0.4152 - val_pr_auc: 0.6854 - val_precision: 0.4614 - val_recall: 0.7719 - val_roc_auc: 0.8869
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 18s 132ms/step - loss: 0.4064 - pr_auc: 0.6882 - precision: 0.4654 - recall: 0.8040 - roc_auc: 0.8963 - val_loss: 0.3326 - val_pr_auc: 0.7733 - val_precision: 0.5068 - val_recall: 0.8433 - val_roc_auc: 0.9294
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 20s 144ms/step - loss: 0.3225 - pr_auc: 0.7654 - precision: 0.5103 - recall: 0.8683 - roc_auc: 0.9349 - val_loss: 0.3719 - val_pr_auc: 0.8170 - val_precision: 0.4776 - val_recall: 0.9619 - val_roc_auc: 0.9527
Epoc

In [14]:
# MODEL 6: TABTRANSFORMER

inputs = Input(shape=(X_train.shape[1],))

x = Reshape((X_train.shape[1], 1))(inputs)

x = Dense(128)(x)

for _ in range(2):

    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=32
    )(x, x)

    x = LayerNormalization()(x + attn)

    ffn = Dense(256, activation="relu")(x)
    ffn = Dropout(0.2)(ffn)
    ffn = Dense(128)(ffn)

    x = LayerNormalization()(x + ffn)

x = GlobalAveragePooling1D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(1, activation='sigmoid')(x)

tabtransformer = Model(inputs, outputs)

tabtransformer.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=fraud_metrics()
)

# Early stopping
es = EarlyStopping(
    monitor='val_pr_auc',
    patience=3,
    restore_best_weights=True,
    mode='max'
)

# Training
tabtransformer_history = tabtransformer.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[es]
)

# Predictions
tabtransformer_val_prob = tabtransformer.predict(X_val, verbose=0).ravel()
tabtransformer_prob = tabtransformer.predict(X_test, verbose=0).ravel()
tabtransformer_pred = finalize_model_outputs("TabTransformer", tabtransformer_val_prob, tabtransformer_prob)

tabtransformer.save(candidate_model_dir / "tabtransformer_model.keras")

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 66s 388ms/step - loss: 0.4424 - pr_auc: 0.5700 - precision: 0.3915 - recall: 0.8259 - roc_auc: 0.8692 - val_loss: 0.3070 - val_pr_auc: 0.7448 - val_precision: 0.4599 - val_recall: 0.9781 - val_roc_auc: 0.9441
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 68s 284ms/step - loss: 0.3937 - pr_auc: 0.6574 - precision: 0.3954 - recall: 0.8775 - roc_auc: 0.8972 - val_loss: 0.6972 - val_pr_auc: 0.6286 - val_precision: 0.2053 - val_recall: 0.9984 - val_roc_auc: 0.8323
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 47s 330ms/step - loss: 0.3200 - pr_auc: 0.7260 - precision: 0.4655 - recall: 0.9222 - roc_auc: 0.9311 - val_loss: 0.2652 - val_pr_auc: 0.8160 - val_precision: 0.5016 - val_recall: 0.9943 - val_roc_auc: 0.9577
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 43s 318ms/step - loss: 0.2730 - pr_auc: 0.7754 - precision: 0.4916 - recall: 0.9612 - roc_auc: 0.9485 - val_loss: 0.2511 - val_pr_auc: 0.8332 - val_precision: 0.5396 - val_recall: 0.9943 - val_roc_auc: 0.9622
Epoc

In [15]:
# SAVE OUTPUTS

np.save(result_dir / "y_test.npy", y_test.to_numpy(dtype=int))
np.save(result_dir / "y_val.npy", y_val.to_numpy(dtype=int))

prediction_frames = []
model_probabilities = {
    "MLP": mlp_prob,
    "GRU": gru_prob,
    "BiLSTM": bilstm_prob,
    "TabNet": tabnet_prob,
    "FT-Transformer": ft_transformer_prob,
    "TabTransformer": tabtransformer_prob,
}
model_predictions = {
    "MLP": mlp_pred,
    "GRU": gru_pred,
    "BiLSTM": bilstm_pred,
    "TabNet": tabnet_pred,
    "FT-Transformer": ft_transformer_pred,
    "TabTransformer": tabtransformer_pred,
}

for model_name, pred in model_predictions.items():
    safe_name = model_name.lower().replace("-", "_").replace(" ", "_")
    prob = model_probabilities[model_name]
    np.save(result_dir / f"{safe_name}_pred.npy", pred.astype(int))
    np.save(result_dir / f"{safe_name}_prob.npy", prob.astype(float))
    prediction_frames.append(
        pd.DataFrame(
            {
                "model": model_name,
                "row_id": np.arange(len(y_test)),
                "y_true": y_test.to_numpy(dtype=int),
                "fraud_probability": prob.astype(float),
                "fraud_prediction": pred.astype(int),
                "decision_threshold": model_thresholds[model_name]["threshold"],
            }
        )
    )

pd.concat(prediction_frames, ignore_index=True).to_csv(
    result_dir / "test_predictions.csv",
    index=False,
)

with open(thresholds_path, "w", encoding="utf-8") as f:
    json.dump(model_thresholds, f, indent=2)

preprocessing_bundle["decision_thresholds"] = model_thresholds
with open(preprocessing_bundle_path, "wb") as f:
    pickle.dump(preprocessing_bundle, f)


In [16]:
# TRAINING HISTORIES AND MANIFEST

def normalize_tabnet_history(tabnet_model):
    if not hasattr(tabnet_model, "history"):
        return None

    history = getattr(tabnet_model.history, "history", tabnet_model.history)
    if not isinstance(history, dict):
        return None

    history = dict(history)

    if "val_loss" not in history:
        for key in ("val_0_logloss", "valid_logloss", "validation_logloss"):
            if key in history:
                history["val_loss"] = history[key]
                break

    if "val_roc_auc" not in history:
        for key in ("val_0_auc", "valid_auc", "validation_auc"):
            if key in history:
                history["val_roc_auc"] = history[key]
                break

    return history


training_histories = {
    "MLP": mlp_history.history,
    "GRU": gru_history.history,
    "BiLSTM": bilstm_history.history,
    "FT-Transformer": ft_history.history,
    "TabTransformer": tabtransformer_history.history,
}

tabnet_history = normalize_tabnet_history(tabnet)
if tabnet_history:
    training_histories["TabNet"] = tabnet_history

with open(artifact_dir / "training_histories.pkl", "wb") as f:
    pickle.dump(training_histories, f)

training_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_data": str(raw_data_path.relative_to(root_path)),
    "candidate_models_dir": str(candidate_model_dir.relative_to(root_path)),
    "models_dir": str(model_dir.relative_to(root_path)),
    "artifacts_dir": str(artifact_dir.relative_to(root_path)),
    "results_dir": str(result_dir.relative_to(root_path)),
    "random_seed": RANDOM_SEED,
    "training_row_count": training_row_count,
    "training_class_distribution": training_class_distribution,
    "dataset_source": "preprocessing_pipeline_downsampled_transactions",
    "feature_columns": features,
    "target_column": "isFraud",
    "split_profile": split_profile,
    "class_weight": class_weight_dict,
    "imbalance_strategy": "pre_downsampled_dataset_with_balanced_class_weights",
    "threshold_selection": {
        "source": "validation_set",
        "objective": "maximize fraud-class F1 subject to minimum recall when feasible",
        "minimum_recall": MIN_RECALL_FOR_FRAUD,
        "thresholds": model_thresholds,
    },
    "primary_metrics": ["recall", "precision", "f1", "roc_auc", "pr_auc"],
}

with open(training_manifest_path, "w", encoding="utf-8") as f:
    json.dump(training_manifest, f, indent=2)

print("Saved training manifest:", training_manifest_path)


Saved training manifest: C:\Project\Transactional-Fraud-Detection\artifacts\training_manifest.json
